# V1 pipeline RAG

## Analyse des besoins et cadre de travail :
- **Cible** : Chercheurs, historiens, archivistes, étudiants en histoire.
- **Cas d'usage** : Recherche d’informations historiques, génération de résumé
- **Types de données** : Articles de journaux historiques, images, métadonnées.
- **Langues** : Français, Anglais
- **Dataset** : Miracl

## Téléchargement des blibliothèques et modèles

In [ ]:
!pip install datasets
!pip install dateparser

!pip install -U pip setuptools wheel
!pip install -U 'spacy[cuda12x]'
!python -m spacy download fr_core_news_sm
!pip install sentence-transformers[onnx-gpu]
!pip install pinecone

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 100.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Import des bibliothèques

In [ ]:
# Prétraitement
import datasets
from datasets import load_dataset
import pandas as pd
from itertools import islice
import re
import dateparser
from dateparser.search import search_dates
import datetime
import spacy
import numpy as np
from joblib import Parallel, delayed

# Embedding et Vectorisation
from sentence_transformers import SentenceTransformer

# Indexation
from pinecone import Pinecone, ServerlessSpec
from tqdm import tqdm
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor

## Prétraitement des données :

### Extraction du contenu brut
1. Téléchargement du dataset
2. Nettoyage du texte brut (suppression des balises HTML, des caractères spéciaux, en-têtes, pieds de page, etc.)

In [ ]:
languages = ['fr', 'en']
max_docs_per_lang = 100000 # taille du dataset (x2 car français + anglais)
all_docs = []

In [ ]:
def load_lang(lang, max_docs):
    print(f"[{lang.upper()}] Chargement...")
    dataset_stream = load_dataset('miracl/miracl-corpus', lang, split='train', streaming=True, trust_remote_code=True)
    sample = islice(dataset_stream, max_docs)

    docs = []
    for doc in sample:
        docs.append({
            "docid": doc["docid"],
            "lang": lang,
            "title": doc["title"],
            "text": doc["text"]
        })
    return docs

In [ ]:
results = Parallel(n_jobs=len(languages))(
    delayed(load_lang)(lang, max_docs_per_lang) for lang in languages
)
all_docs = [doc for lang_docs in results for doc in lang_docs]

In [ ]:
df = pd.DataFrame(all_docs)

In [ ]:
df.head()

,docid,lang,title,text
0,3#0,fr,Antoine Meillet,"Paul Jules Antoine Meillet, né le à Moulins (A..."
1,3#1,fr,Antoine Meillet,"D'origine bourbonnaise, fils d'un notaire de C..."
2,3#2,fr,Antoine Meillet,Étudiant à la faculté des lettres de Paris à p...
3,3#3,fr,Antoine Meillet,"En 1889, il est major de l'agrégation de gramm..."
4,3#4,fr,Antoine Meillet,Il assure à la suite de Saussure le cours de g...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   docid   200000 non-null  object
 1   lang    200000 non-null  object
 2   title   200000 non-null  object
 3   text    200000 non-null  object
dtypes: object(4)
memory usage: 6.1+ MB


In [ ]:
def clean_text(text):
  text = re.sub(r"<[^>]+>", "", text)  # Supprimer les balises HTML
  # text = re.sub(r"[^\w\s,.?!]", "", text)  # Supprimer caractères spéciaux
  text = re.sub(r"\s+", " ", text)  # Supprimer espaces multiples
  return text.strip()

In [ ]:
df["title"] = df["title"].apply(clean_text)
df["text"] = df["text"].apply(clean_text)

In [ ]:
df.describe()

,docid,lang,title,text
count,200000,200000,200000,200000
unique,180951,2,6123,198239
top,5288#19,fr,Strasbourg,Et aussi aux
freq,2,100000,423,178


### Segmentation des articles
1. Chunking sémantique (Déjà fait par le corpus)
2. Limitation de la taille des chunks (512 tokens)
3. Ajout de métadonnées (date, titre, auteur, etc.)

In [ ]:
# chunk_size = 512 # Pas besoin ici
chunk_text = []
chunk_meta = []

In [ ]:
for index, row in df.iterrows():
    text = row["text"]
    title = row["title"]
    docid = row["docid"]
    lang = row["lang"]
    sentences = text.split(". ")

    for sentence in sentences:
        chunk_text.append(sentence)
        chunk_meta.append({
            "title": title,
            "docid": docid,
            "lang": lang
        })

### Nettoyage et normalisation
1. Suppression des doublons
2. Normalisation des dates
3. Vérification de la langue

In [ ]:
print(len(chunk_text))
print(len(chunk_meta))

675198
675198


In [ ]:
# Suppression des doublons
df_chunk = pd.DataFrame({"text": chunk_text, "meta": chunk_meta})
df_chunk = df_chunk.drop_duplicates(subset=["text"])

In [ ]:
# Normalisation des dates dans le text
def extract_dates(text, lang):
    results = search_dates(text, languages=[lang], settings={'STRICT_PARSING': True})
    if not results:
        return [], None, None

    iso_dates = []
    for _, date in results:
        try:
            iso_dates.append(date.date().isoformat())
        except Exception:
            continue

    # Trier et déterminer min/max
    iso_dates = sorted(set(iso_dates))
    earliest = min(iso_dates) if iso_dates else None
    latest = max(iso_dates) if iso_dates else None

    return iso_dates, earliest, latest

In [ ]:
def enrich_meta_with_dates(row):
    iso_dates, earliest, latest = extract_dates(row["text"], row["meta"]["lang"])
    meta = row["meta"].copy() if isinstance(row["meta"], dict) else {}

    meta["dates_iso"] = iso_dates
    meta["earliest_date"] = earliest
    meta["latest_date"] = latest
    return meta

In [ ]:
def parallel_enrich_meta(df_chunk, n_jobs=-1):
    rows = df_chunk.to_dict(orient="records")

    metas = Parallel(n_jobs=8)(
      delayed(enrich_meta_with_dates)(row) for row in tqdm(rows)
    )

    df_chunk["meta"] = metas
    return df_chunk

In [ ]:
df_chunk = parallel_enrich_meta(df_chunk, n_jobs=32)


100%|██████████| 667270/667270 [02:06<00:00, 5259.42it/s]


In [ ]:
df_chunk.head() # Amélioration possible avec les dates (uniquement année)

,text,meta
0,"Paul Jules Antoine Meillet, né le à Moulins Al...","{'title': 'Antoine Meillet', 'docid': '3#0', '..."
1,Il est aussi philologue.,"{'title': 'Antoine Meillet', 'docid': '3#0', '..."
2,"Dorigine bourbonnaise, fils dun notaire de Châ...","{'title': 'Antoine Meillet', 'docid': '3#1', '..."
3,Étudiant à la faculté des lettres de Paris à p...,"{'title': 'Antoine Meillet', 'docid': '3#2', '..."
4,"En 1889, il est major de lagrégation de gramma...","{'title': 'Antoine Meillet', 'docid': '3#3', '..."


### Reconnaissance des éntités nommées :


In [ ]:
nlp_fr = spacy.load("fr_core_news_sm", disable=["tagger", "parser"])
nlp_en = spacy.load("en_core_web_sm", disable=["tagger", "parser"])

In [ ]:
def enrich_df_with_ner_pipe(df_chunk): # À AMÉLIORER
    enriched_rows = []

    for lang in ["fr", "en"]:
        sub_df = df_chunk[df_chunk["meta"].apply(lambda m: m.get("lang") == lang)]
        if sub_df.empty:
            continue

        nlp = get_nlp(lang)
        texts = sub_df["text"].tolist()
        metas = sub_df["meta"].tolist()

        docs = list(nlp.pipe(texts, batch_size=256))

        for meta, doc in zip(metas, docs):
            entities = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]
            meta = meta.copy()
            meta["entities"] = entities
            enriched_rows.append({"text": doc.text, "meta": meta})

    return pd.DataFrame(enriched_rows)

In [ ]:
chunks = np.array_split(df_chunk, 64)

/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
_nlp_models = {}

def get_nlp(lang):
    if lang not in _nlp_models:
        _nlp_models[lang] = spacy.load(
            "fr_core_news_sm" if lang == "fr" else "en_core_web_sm",
            disable=["tagger", "parser"]
        )
    return _nlp_models[lang]

In [ ]:
results = Parallel(n_jobs=8, backend="loky", prefer="processes")(
    delayed(enrich_df_with_ner_pipe)(chunk) for chunk in tqdm(chunks)
)


 25%|██▌       | 16/64 [00:54<02:42,  3.38s/it]/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeou

In [ ]:
df = pd.concat(results, ignore_index=True)

In [ ]:
df_chunk = df
df_chunk.head()

,text,meta
0,"Paul Jules Antoine Meillet, né le à Moulins Al...","{'title': 'Antoine Meillet', 'docid': '3#0', '..."
1,Il est aussi philologue.,"{'title': 'Antoine Meillet', 'docid': '3#0', '..."
2,"Dorigine bourbonnaise, fils dun notaire de Châ...","{'title': 'Antoine Meillet', 'docid': '3#1', '..."
3,Étudiant à la faculté des lettres de Paris à p...,"{'title': 'Antoine Meillet', 'docid': '3#2', '..."
4,"En 1889, il est major de lagrégation de gramma...","{'title': 'Antoine Meillet', 'docid': '3#3', '..."


## Embedding et Vectorisation :

### Choix du modèle d'embedding
1. [gte-Qwen2-1.5B-instruct](https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct) -> Trop gros
2. [e5-small-v2](https://huggingface.co/intfloat/e5-small-v2)

In [ ]:
embedding_model = SentenceTransformer("intfloat/e5-small-v2", trust_remote_code=True, backend="onnx")

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:505 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************


### Vectorisation des chunks
1. Transformation des chunks en vecteurs d'embedding

In [ ]:
df["chunk"] = df["meta"].apply(lambda m: m.get("title", "")) + "\n" + df["text"]

In [ ]:
texts = df["chunk"].tolist()

In [ ]:
embeddings = embedding_model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/2607 [00:00<?, ?it/s]

### Sauvegarde des vecteurs

In [ ]:
np_embeddings = np.array(embeddings)
np.save("embeddings.npy", np_embeddings)

In [ ]:
len(embeddings)

667270

## Indexation

In [ ]:
embeddings = np.load("embeddings.npy")

### Création de la base de donnée vectorielle

In [ ]:
pc = Pinecone(api_key=userdata.get("PINECONE"))

In [ ]:
index_name = "rag-v1"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=embeddings.shape[1],
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [ ]:
index = pc.Index(index_name)

### Création du set de donnée

Chaque entrée a :
- un identifiant
- un vecteur
- le text d'origine
- les éléments de métadonnée

In [ ]:
def format_for_pinecone(i, row, embedding):
    meta = row["meta"]
    return (
        str(i),
        embedding.astype("float32").tolist(),
        {
            "title": meta["title"],
            "lang": meta["lang"],
            "year": meta["earliest_date"][:4] if meta["earliest_date"] else "",
            "entities": [ent["text"] for ent in meta.get("entities", [])][:5]
        }
    )

In [ ]:
pinecone_data_parallel = Parallel(n_jobs=-1)(
    delayed(format_for_pinecone)(i, row, embeddings[i])
    for i, row in tqdm(enumerate(df_chunk.to_dict(orient="records")), total=len(df_chunk))
)



  0%|          | 0/667270 [00:00<?, ?it/s]

  0%|          | 12/667270 [00:05<87:23:52,  2.12it/s]

  0%|          | 96/667270 [00:05<8:10:21, 22.68it/s] 

  0%|          | 768/667270 [00:05<44:41, 248.52it/s]

  0%|          | 2304/667270 [00:05<11:48, 938.50it/s]

  1%|          | 4608/667270 [00:06<04:51, 2270.67it/s]

  1%|          | 7771/667270 [00:06<02:23, 4606.69it/s]

  1%|▏         | 9602/667270 [00:06<01:50, 5951.65it/s]

  2%|▏         | 12288/667270 [00:06<01:19, 8192.39it/s]

  3%|▎         | 18432/667270 [00:06<00:47, 13796.02it/s]

  4%|▎         | 24576/667270 [00:06<00:38, 16807.09it/s]

  5%|▍         | 30720/667270 [00:07<00:37, 16930.46it/s]

  6%|▌         | 36864/667270 [00:07<00:37, 17034.47it/s]

  6%|▋         | 43008/667270 [00:08<00:36, 17048.28it/s]

  7%|▋         | 49152/667270 [00:08<00:36, 17156.42it/s]

  8%|▊         | 55296/667270 [00:08<00:35, 17251.90it/s]

  9%|▉         | 61440/667270 [00:09<00:34, 17364.77it/s]

 10%|█         | 67584/667270 

### Insertion dans la base de donnée

In [ ]:
def insert_batches(index, pinecone_data, batch_size=256, max_workers=4):
    def insert_batch(batch):
        index.upsert(vectors=batch, show_progress=True)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in range(0, len(pinecone_data), batch_size):
            batch = pinecone_data[i:i+batch_size]
            futures.append(executor.submit(insert_batch, batch))

In [ ]:
insert_batches(index, pinecone_data_parallel, batch_size=256, max_workers=16)

### Test de recherche

In [ ]:
query_text = "Qui est le plus grand voleur français ?"

In [ ]:
query_vector = embedding_model.encode([query_text], normalize_embeddings=True)

In [ ]:
results = index.query(
    vector=query_vector.astype("float32").tolist(),
    top_k=5,
    include_metadata=True
)

In [ ]:
print(results)

{'matches': [{'id': '119731',
              'metadata': {'entities': [],
                           'lang': 'fr',
                           'title': 'Français',
                           'year': ''},
              'score': 0.870938063,
              'values': []},
             {'id': '119608',
              'metadata': {'entities': [],
                           'lang': 'fr',
                           'title': 'Français',
                           'year': ''},
              'score': 0.870518923,
              'values': []},
             {'id': '119733',
              'metadata': {'entities': [],
                           'lang': 'fr',
                           'title': 'Français',
                           'year': ''},
              'score': 0.870498598,
              'values': []},
             {'id': '119447',
              'metadata': {'entities': ['ÉtatsUnis'],
                           'lang': 'fr',
                           'title': 'Français',
                          

In [ ]:
for match in results["matches"]:
    doc_id = int(match["id"])
    print(df.iloc[doc_id]["text"])

De façon générale, le français demeure une des langues les plus enseignées dans le monde.
Le français est la deuxième langue la plus fréquemment utilisée dans les rencontres internationales.
Le français est enseigné de manière rudimentaire simples notions dorthographe et de grammaire
Le français est la deuxième langue la plus souvent enseignée en tant que langue étrangère à travers le monde, y compris aux ÉtatsUnis
Au long du , le français simpose comme langue scientifique et comme langue denseignement


## Récupération d'informations :


In [ ]:
basic_query = "Que s'est il passé lors de l'année 1956 en France ou dans le monde ?"

query_vector = embedding_model.encode([query_text], normalize_embeddings=True)

### Récupération des chunks les plus pertinents

In [ ]:
top_k_limit = 50

In [ ]:
results = index.query(
    vector=query_vector.astype("float32").tolist(),
    top_k=top_k_limit,
    include_metadata=True
)

### Première évaluation du retrieval

### Recherche sémantique
1. Utilisation de Milvus pour la recherche de vecteurs similaires
2. Récupération des chunks les plus pertinents (Top-k Retrieval)
3. Re-ranking des résultats
4. Hybrid Search (Combine recherche sémantique (vecteurs) et recherche classique (BM25).)
5. Traduction des résultats dans une langue demandé

## Génération de réponses :

### Query

### 1er retrival

### Reranking

### Génération de la réponse